# Session 2: the mathematics of a gear train
**A. Real-world problem.** A motor spins at 1200 rpm but a conveyor must turn at 60 rpm.
We need a reduction of 1:20 from gears that actually exist: whole numbers of teeth, no
pinion so small that its teeth are cut away, and a size that fits in a box.

**Plan.** (1) stage and compound ratios, (2) speed, torque and power, (3) module, pitch
diameter and centre distance, (4) the smallest integer-tooth design for 1:20.

In [ ]:
import pathlib
from fractions import Fraction

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: ratios multiply
For one mesh the *speed ratio* is `i = N_driven / N_driver` (more than 1 means a reduction).
A compound train has several stages; two gears sit on the same shaft (so they turn together),
and the stage ratios multiply: `i = (N2/N1) * (N4/N3)`. The output speed is `w_in / i`, the
ideal output torque is `T_in * i`, and the power `T * w` is unchanged (ideal = no losses).
The direction after k external meshes is `(-1)^k`.

In [ ]:
def stage_ratio(n_driver, n_driven):
    return n_driven / n_driver


def compound_ratio(stages):
    """Overall speed ratio of a train given as [(N_driver, N_driven), ...]."""
    i = 1.0
    for n_driver, n_driven in stages:
        i *= stage_ratio(n_driver, n_driven)
    return i


def output_speed(w_in, stages):
    """Output speed (same unit as w_in); the sign is the direction, one external mesh per stage."""
    return (-1) ** len(stages) * w_in / compound_ratio(stages)


def output_torque_ideal(t_in, stages):
    """Ideal output torque (no losses), same sign convention as the speed."""
    return (-1) ** len(stages) * t_in * compound_ratio(stages)

## Step 2: module, pitch diameter and centre distance
Gears that mesh share the same tooth size, the *module* m (mm per tooth). The pitch circle
of a gear with N teeth has diameter `d = m N`, and two meshing gears sit a centre distance
`a = (d1 + d2) / 2 = m (N1 + N2) / 2` apart.

In [ ]:
def pitch_diameter(module_mm, n_teeth):
    return module_mm * n_teeth


def centre_distance(module_mm, n1, n2):
    return module_mm * (n1 + n2) / 2

## Step 3: the minimum number of teeth, and the contact ratio
A gear cutter can remove the root of a tooth of a small pinion (*undercutting*). For the
common 20 degree pressure angle the limit from geometry is `2 / sin^2(phi)` = 17.1 teeth, so
17 teeth is a commonly used practical guideline for a pinion. Meshing also needs the *contact
ratio* (the average number of tooth pairs in contact) to be above 1, or the drive would
jump from tooth to tooth.

In [ ]:
def min_teeth_no_undercut(phi_deg=20.0):
    return 2.0 / np.sin(np.radians(phi_deg)) ** 2


def contact_ratio(n1, n2, phi_deg=20.0):
    """Contact ratio of a standard full-depth spur pair (addendum = module)."""
    phi = np.radians(phi_deg)
    r1, r2 = n1 / 2.0, n2 / 2.0              # pitch radii for module 1
    ra1, ra2 = r1 + 1.0, r2 + 1.0            # addendum radii
    rb1, rb2 = r1 * np.cos(phi), r2 * np.cos(phi)
    path = np.sqrt(ra1 ** 2 - rb1 ** 2) + np.sqrt(ra2 ** 2 - rb2 ** 2) - (r1 + r2) * np.sin(phi)
    return float(path / (np.pi * np.cos(phi)))

## Step 4: design a 1:20 reduction with the fewest teeth
One stage would need a 17-tooth pinion and a 340-tooth gear. Two compound stages are better.
We list every pair (N_driver, N_driven) with both counts from 17 to 100, group them by their
exact ratio (a `Fraction`, so no rounding), and join two groups whose ratios multiply to 20.

In [ ]:
def fewest_teeth_designs(target, n_min=17, n_max=100):
    """All two-stage integer designs with ratio exactly `target`, smallest total teeth first.

    Each entry is (total_teeth, (N1, N2), (N3, N4)).
    """
    groups = {}
    for a in range(n_min, n_max + 1):
        for b in range(n_min, n_max + 1):
            groups.setdefault(Fraction(b, a), []).append((a, b))
    found = []
    for r, first in groups.items():
        second = groups.get(Fraction(target) / r)
        if second:
            for p in first:
                for q in second:
                    found.append((sum(p) + sum(q), p, q))
    found.sort()
    return found


def shaft_speeds(w_in, stages):
    """Speed of each shaft: input, then after each stage (signed)."""
    out = [w_in]
    for n_driver, n_driven in stages:
        out.append(-out[-1] * n_driver / n_driven)
    return out

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    designs = fewest_teeth_designs(20)
    best = designs[0]
    print("number of exact 1:20 two-stage designs (17..100 teeth):", len(designs))
    print("smallest total:", best)
    stages = [best[1], best[2]]
    print("overall ratio", compound_ratio(stages), "speeds", shaft_speeds(1200, stages))
    print("undercut limit", round(min_teeth_no_undercut(), 2))
    print("contact ratio 20:40", round(contact_ratio(20, 40), 3), " 17:85", round(contact_ratio(17, 85), 3))
    print("centre distances (module 2):", centre_distance(2, 17, 68), centre_distance(2, 17, 85), "mm")

    # figure 1: all pair ratios and the 1:20 hyperbola
    n = np.arange(17, 101)
    fig, ax = plt.subplots(figsize=(6, 4.2))
    r1 = np.linspace(1.0, 6.0, 200)
    ax.plot(r1, 20 / r1, color="tab:blue", label="stage 1 x stage 2 = 20")
    for tot, p, q in designs[:12]:
        ax.plot(p[1] / p[0], q[1] / q[0], "o", color="tab:red", ms=4)
    ax.plot([], [], "o", color="tab:red", label="integer designs (smallest 12)")
    ax.set_xlabel("stage 1 ratio N2 / N1")
    ax.set_ylabel("stage 2 ratio N4 / N3")
    ax.set_title("Two stages whose ratios multiply to 20")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "ratio_hyperbola.png", dpi=120)

    # figure 2: speed and torque along the shafts (power stays constant)
    w = np.abs(shaft_speeds(1200, stages))
    t = 2.0 * np.array([1.0, stages[0][1] / stages[0][0], compound_ratio(stages)])
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot([0, 1, 2], w, "o-", label="speed (rpm)")
    ax2.set_xticks([0, 1, 2], ["input shaft", "middle shaft", "output shaft"])
    ax2.set_ylabel("speed (rpm)")
    ax3 = ax2.twinx()
    ax3.plot([0, 1, 2], t, "s-", color="tab:red")
    ax3.set_ylabel("ideal torque (N m)")
    ax2.set_title("Speed falls, torque rises: power is the same")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "shaft_speed_torque.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Change `n_max` or the target, predict how many designs there will be, then run it. Answer the
four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Change the target in `fewest_teeth_designs` from 20 to 12 and print the best design. Predict it first (see question D).
2. **Stretch.** Add a function `total_teeth(design)` and sort the 146 designs for the target 20 by the largest gear instead of by total teeth. Does the best design change?
3. **Explain.** Explain why the stage ratios multiply but the speeds on the same shaft are equal.

## Assessment
- **A. Conceptual.** Why is a two-stage train better than one pair of gears for a 1:20 reduction? Give two reasons.
- **B. Numerical.** A train has stages 20-50 and 18-72. Find the overall ratio, the output speed and the ideal output torque for 900 rpm and 3 N m in, and the direction.
- **C. Graph and simulation interpretation.** In the ratio graph with N1 = 20, N2 = 60 and N3 = 20, which N4 gives a ratio of 20? Can you set that on the slider?
- **D. Design and optimisation challenge.** Design: find the exact two-stage reduction of 1:12 with 17 to 100 teeth and the fewest total teeth. Give the centre distances at module 2 mm.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    designs = L.fewest_teeth_designs(20)
    best = designs[0]
    stages = [best[1], best[2]]
    checks.append(("worked example: the smallest exact 1:20 design is 17-68 then 17-85 (187 teeth)",
                   best[0] == 187 and set(stages) == {(17, 68), (17, 85)}))
    checks.append(("worked example: its stage ratios are 4 and 5 and the overall ratio is exactly 20",
                   sorted([L.stage_ratio(*s) for s in stages]) == [4.0, 5.0] and abs(L.compound_ratio(stages) - 20.0) < 1e-12))
    checks.append(("worked example: 1200 rpm in gives 60 rpm out, same direction (two meshes)",
                   abs(L.output_speed(1200, stages) - 60.0) < 1e-9))
    checks.append(("worked example: 2 N m in gives 40 N m out for the ideal train",
                   abs(L.output_torque_ideal(2.0, stages) - 40.0) < 1e-9))
    sp = L.shaft_speeds(1200, stages)
    checks.append(("middle shaft turns at 300 rpm the other way (1200 / 4 with the 17-68 stage first)",
                   abs(sp[1] + 300.0) < 1e-9))
    checks.append(("a single stage for 1:20 with a 17-tooth pinion needs 340 teeth",
                   abs(L.stage_ratio(17, 340) - 20.0) < 1e-12))
    p_in = 2.0 * 1200
    p_out = L.output_torque_ideal(2.0, stages) * abs(L.output_speed(1200, stages))
    checks.append(("power T x w is unchanged by the ideal train", abs(p_in - p_out) < 1e-6))
    checks.append(("the number of exact two-stage designs with 17 to 100 teeth is 146", len(designs) == 146))
    coax = [d for d in designs if sum(d[1]) == sum(d[2])]
    checks.append(("only 4 of those 146 have equal centre distances (same module)", len(coax) == 4))
    checks.append(("centre distances at module 2 mm: 85 mm and 102 mm, which are not equal",
                   L.centre_distance(2, 17, 68) == 85.0 and L.centre_distance(2, 17, 85) == 102.0))
    checks.append(("pitch diameter d = m N: module 2 and 17 teeth gives 34 mm", L.pitch_diameter(2, 17) == 34))
    checks.append(("undercut limit 2 / sin^2(20 degrees) = 17.1", abs(L.min_teeth_no_undercut() - 17.097) < 1e-3))
    c = L.contact_ratio(20, 40)
    checks.append(("contact ratio of a 20:40 pair is 1.635 (above 1)", abs(c - 1.635) < 1e-3 and c > 1.0))
    checks.append(("contact ratio of a 17:17 pair is 1.515, smaller than for 100:100 (1.853)",
                   abs(L.contact_ratio(17, 17) - 1.515) < 1e-3 and L.contact_ratio(17, 17) < L.contact_ratio(100, 100)))
    checks.append(("middle shaft ideal torque is 8 N m (2 N m x 4)", abs(abs(L.output_torque_ideal(2.0, [stages[0]])) - 8.0) < 1e-9))
    s_b = [(20, 50), (18, 72)]
    checks.append(("hand calculation: 20-50 then 18-72 at 900 rpm and 3 N m gives 90 rpm and 30 N m, same direction",
                   abs(L.compound_ratio(s_b) - 10.0) < 1e-12 and abs(L.output_speed(900, s_b) - 90.0) < 1e-9
                   and abs(L.output_torque_ideal(3.0, s_b) - 30.0) < 1e-9))
    checks.append(("graph reading: with N1 = N3 = 20 and N2 = 60 the ratio 20 needs N4 = 133.3 (more than 100)",
                   abs(20 * 20 * 20 / 60 - 133.33) < 0.01))
    d12 = L.fewest_teeth_designs(12)
    checks.append(("design problem: the smallest exact 1:12 design is 17-51 then 17-68 (153 teeth)",
                   d12[0][0] == 153 and set(d12[0][1:]) == {(17, 51), (17, 68)}))
    checks.append(("module 2 mm: a 340-tooth gear is 680 mm across, an 85-tooth gear 170 mm",
                   L.pitch_diameter(2, 340) == 680 and L.pitch_diameter(2, 85) == 170))
    checks.append(("1:12 design at module 2 mm: centre distances 68 mm and 85 mm",
                   L.centre_distance(2, 17, 51) == 68.0 and L.centre_distance(2, 17, 68) == 85.0))
    checks.append(("direction: three stages (three meshes) turn the output the other way",
                   L.output_speed(100, [(20, 40)] * 3) < 0))
    checks.append(("edge case: equal teeth give ratio exactly 1", L.compound_ratio([(30, 30), (25, 25)]) == 1.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
